In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

# ==============================================================================
# CELL 1: GAUSSIAN PULSE VARIATIONS (5 VARIATIONS IN INCREASING TIME ORDER)
# ==============================================================================

# ----------------------------- physical constants ---------------------------
hbar = 6.582119569e-4          # eV * ps

# ------------------------- device D5 parameters -----------------------------
Delta1, Delta2, DeltaT, Deltac = 1.80, 1.98, 1.20, 1.82   # eV (CT removed)
g_bare = 500e-9                # eV
N      = 5.0e10                
G      = g_bare * np.sqrt(N)   # collective coupling (eV)

kappa, gamma_m, gamma_T = 29.0, 0.5, 0.01     # ps^-1
gamma_z, gamma_ISC      = 20.0, 5.0           # ps^-1

# Pulse parameters
r          = 0.5
t0         = 0.10                                       # ps
sigma_orig = (35.0 / (2*np.sqrt(2*np.log(2)))) * 1e-3   # ~0.01486 ps (FWHM = 35.0 fs)

# 5 Gaussian pulse width variations (ordered by increasing pulse duration)
# Keeping the 3 original variations (narrow, orig, wide) + 1 in-between + 1 wider
sigma_narrow    = sigma_orig * 0.5    # 1. Narrow Gaussian    (FWHM = 17.5 fs) [Original Narrow]
# sigma_orig    = sigma_orig * 1.0    # 2. Original Baseline  (FWHM = 35.0 fs) [Original Baseline]
sigma_medium    = sigma_orig * 1.75   # 3. Medium Gaussian    (FWHM = 61.25 fs) [New - In-between]
sigma_wide      = sigma_orig * 2.5    # 4. Wide Gaussian      (FWHM = 87.5 fs) [Original Wide]
sigma_very_wide = sigma_orig * 3.5    # 5. Very Wide Gaussian (FWHM = 122.5 fs) [New - Wider]

# Reference total pulse energy (photon fluence): I0 = \int_{-\infty}^{\infty} |\eta_{orig}(t)|^2 dt
# Karena \eta_{orig}(t) = (\sqrt{r} / (\sigma_{orig}\sqrt{2\pi})) * exp(-(t-t0)^2 / (2\sigma_{orig}^2)),
# maka I0 = r / (2 * \sqrt{\pi} * \sigma_{orig})
I0 = r / (2.0 * np.sqrt(np.pi) * sigma_orig)            # ~9.4934 ps^-1

M_pol = np.array([[Deltac, G,      G     ],
                  [G,      Delta1, 0.0   ],
                  [G,      0.0,    Delta2]])
nu = np.linalg.eigvalsh(M_pol)[0]

d1, d2, dc = (Delta1 - nu)/hbar, (Delta2 - nu)/hbar, (Deltac - nu)/hbar
Gw = G / hbar

def build_model():
    """Operators, Hamiltonian, collapse and measurement operators (without CT)."""
    Na, Nb = 5, 3              
    Ia, Ib, I2 = qt.qeye(Na), qt.qeye(Nb), qt.qeye(2)

    def emb(op, slot):
        ops = [Ia, Ib, Ib, I2, I2, I2]
        ops[slot] = op
        return qt.tensor(*ops)

    a   = emb(qt.destroy(Na), 0)            
    b1  = emb(qt.destroy(Nb), 1)            
    b2  = emb(qt.destroy(Nb), 2)            
    dk1 = emb(qt.destroy(2), 3)            
    dk2 = emb(qt.destroy(2), 4)            
    Tm  = emb(qt.destroy(2), 5)            
    
    H0 = (dc * a.dag()*a + d1 * b1.dag()*b1 + d2 * b2.dag()*b2
          + Gw * (a.dag()*(b1 + b2) + (b1.dag() + b2.dag())*a))
    Hdrive = 1j * (a.dag() - a)            

    c_ops = [np.sqrt(kappa)     * a,
             np.sqrt(gamma_m)   * b1,               
             np.sqrt(gamma_m)   * b2,               
             np.sqrt(gamma_m)   * dk1,              
             np.sqrt(gamma_m)   * dk2,              
             np.sqrt(gamma_T)   * Tm,               
             np.sqrt(gamma_z)   * dk1.dag()*b1,     
             np.sqrt(gamma_z)   * dk2.dag()*b2,
             np.sqrt(gamma_ISC) * Tm.dag()*b1,      
             np.sqrt(gamma_ISC) * Tm.dag()*b2,
             np.sqrt(gamma_ISC) * Tm.dag()*dk1,     
             np.sqrt(gamma_ISC) * Tm.dag()*dk2]             

    e_ops = [b1.dag()*b1 + dk1.dag()*dk1,           # <X11>  (S1^0)
             b2.dag()*b2 + dk2.dag()*dk2,           # <X22>  (S1^1)
             Tm.dag()*Tm,                           # <XTT>  (T1)
             a.dag()*a]                             # photons per molecule

    H_batt = (Delta1 * (b1.dag()*b1 + dk1.dag()*dk1)
              + Delta2 * (b2.dag()*b2 + dk2.dag()*dk2)
              + DeltaT * Tm.dag()*Tm)

    # Molecular battery Hamiltonian (tracing out cavity slot 0) for ergotropy calculation
    b1_mol  = qt.tensor(qt.destroy(Nb), Ib, I2, I2, I2)
    b2_mol  = qt.tensor(Ib, qt.destroy(Nb), I2, I2, I2)
    dk1_mol = qt.tensor(Ib, Ib, qt.destroy(2), I2, I2)
    dk2_mol = qt.tensor(Ib, Ib, I2, qt.destroy(2), I2)
    Tm_mol  = qt.tensor(Ib, Ib, I2, I2, qt.destroy(2))
    H_batt_mol = (Delta1 * (b1_mol.dag()*b1_mol + dk1_mol.dag()*dk1_mol)
                  + Delta2 * (b2_mol.dag()*b2_mol + dk2_mol.dag()*dk2_mol)
                  + DeltaT * Tm_mol.dag()*Tm_mol)

    psi0 = qt.tensor(qt.basis(Na, 0), qt.basis(Nb, 0), qt.basis(Nb, 0),
                     qt.basis(2, 0), qt.basis(2, 0), qt.basis(2, 0))
    return H0, Hdrive, c_ops, e_ops, psi0, H_batt, H_batt_mol


def compute_thermodynamics(res, tlist, eps_sorted):
    """Compute stored energy E(t), charging power P(t), and ergotropy E_ext(t)."""
    P_S10 = np.real(res.expect[0])
    P_S11 = np.real(res.expect[1])
    P_T1  = np.real(res.expect[2])
    
    # Stored energy E(t) = <H_batt> (in eV)
    E_stored = Delta1 * P_S10 + Delta2 * P_S11 + DeltaT * P_T1
    # Instantaneous charging power P(t) = dE(t)/dt (in eV/ps)
    P_charge = np.gradient(E_stored, tlist)
    
    # Ergotropy (extractable work) of the molecular battery subsystem:
    # W_ext(t) = Tr[rho_mol(t) H_batt_mol] - sum_k r_k(t) * eps_k
    ergotropy = np.zeros(len(tlist))
    for i, state in enumerate(res.states):
        rho_mol = state.ptrace([1, 2, 3, 4, 5])
        r_vals = np.sort(np.real(rho_mol.eigenenergies()))[::-1]
        ergotropy[i] = max(0.0, E_stored[i] - np.sum(r_vals * eps_sorted))
        
    return E_stored, P_charge, ergotropy


# Gaussian Drive functions (all normalized to the same total pulse energy \int |\eta(t)|^2 dt = I0)
def eta_narrow(t, **kwargs):
    return np.sqrt(I0 / (sigma_narrow * np.sqrt(np.pi))) * np.exp(-(t - t0)**2 / (2*sigma_narrow**2))

def eta_orig(t, **kwargs):
    return np.sqrt(I0 / (sigma_orig * np.sqrt(np.pi))) * np.exp(-(t - t0)**2 / (2*sigma_orig**2))

def eta_medium(t, **kwargs):
    return np.sqrt(I0 / (sigma_medium * np.sqrt(np.pi))) * np.exp(-(t - t0)**2 / (2*sigma_medium**2))

def eta_wide(t, **kwargs):
    return np.sqrt(I0 / (sigma_wide * np.sqrt(np.pi))) * np.exp(-(t - t0)**2 / (2*sigma_wide**2))

def eta_very_wide(t, **kwargs):
    return np.sqrt(I0 / (sigma_very_wide * np.sqrt(np.pi))) * np.exp(-(t - t0)**2 / (2*sigma_very_wide**2))


H0, Hdrive, c_ops, e_ops, psi0, H_batt, H_batt_mol = build_model()
eps_batt_sorted = np.sort(np.real(H_batt_mol.eigenenergies()))

tlist = np.linspace(0.0, 4.0, 801)     # ps
# max_step wajib: tanpa ini solver adaptif bisa "melompati" pulsa sempit (sigma ~7 fs)
opts = {"atol": 1e-9, "rtol": 1e-7, "nsteps": 1000000, "store_states": True,
        "max_step": sigma_narrow / 5}   # ~1.5 fs

print("Running 1/5: Narrow Gaussian (0.5x sigma_orig)...")
res_narrow = qt.mesolve([H0, [Hdrive, eta_narrow]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 2/5: Original Baseline Gaussian (1.0x sigma_orig)...")
res_orig = qt.mesolve([H0, [Hdrive, eta_orig]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 3/5: Medium Gaussian (1.75x sigma_orig)...")
res_medium = qt.mesolve([H0, [Hdrive, eta_medium]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 4/5: Wide Gaussian (2.5x sigma_orig)...")
res_wide = qt.mesolve([H0, [Hdrive, eta_wide]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 5/5: Very Wide Gaussian (3.5x sigma_orig)...")
res_very_wide = qt.mesolve([H0, [Hdrive, eta_very_wide]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Computing thermodynamic observables for Gaussian variations...")
thermo_narrow    = compute_thermodynamics(res_narrow,    tlist, eps_batt_sorted)
thermo_orig      = compute_thermodynamics(res_orig,      tlist, eps_batt_sorted)
thermo_medium    = compute_thermodynamics(res_medium,    tlist, eps_batt_sorted)
thermo_wide      = compute_thermodynamics(res_wide,      tlist, eps_batt_sorted)
thermo_very_wide = compute_thermodynamics(res_very_wide, tlist, eps_batt_sorted)

# ------------------------------------------------------------------------------
# 1. PLOT GAUSSIAN LASER PULSE PROFILES
# ------------------------------------------------------------------------------
print("Plotting Gaussian laser pulse profiles...")
t_pulse_g = np.linspace(0.0, 0.40, 1000)
fwhm_factor = 2.0 * np.sqrt(2.0 * np.log(2.0)) * 1e3

gaussian_pulse_list = [
    ("1. Narrow Gaussian",     [eta_narrow(t) for t in t_pulse_g],    "#2ca02c", f"FWHM = {sigma_narrow*fwhm_factor:.1f} fs ($\\sigma = {sigma_narrow*1e3:.1f}$ fs)"),
    ("2. Original (Baseline)", [eta_orig(t) for t in t_pulse_g],      "#d62728", f"FWHM = {sigma_orig*fwhm_factor:.1f} fs ($\\sigma = {sigma_orig*1e3:.1f}$ fs)"),
    ("3. Medium Gaussian",     [eta_medium(t) for t in t_pulse_g],    "#ff7f0e", f"FWHM = {sigma_medium*fwhm_factor:.1f} fs ($\\sigma = {sigma_medium*1e3:.1f}$ fs)"),
    ("4. Wide Gaussian",       [eta_wide(t) for t in t_pulse_g],      "#1f77b4", f"FWHM = {sigma_wide*fwhm_factor:.1f} fs ($\\sigma = {sigma_wide*1e3:.1f}$ fs)"),
    ("5. Very Wide Gaussian",  [eta_very_wide(t) for t in t_pulse_g], "#9467bd", f"FWHM = {sigma_very_wide*fwhm_factor:.1f} fs ($\\sigma = {sigma_very_wide*1e3:.1f}$ fs)"),
]

fig_pulse_g, axs_pg = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)
fig_pulse_g.suptitle(r"Gaussian Laser Pulse Profiles $\eta(t)$ (Energy-Normalized: $\int |\eta(t)|^2\,dt = I_0$)", y=0.98)

for ax, (title, y_vals, color, label_str) in zip(axs_pg.flat[:5], gaussian_pulse_list):
    y_arr = np.array(y_vals)
    ax.plot(t_pulse_g, y_arr, color=color, lw=2.3, label=label_str)
    ax.fill_between(t_pulse_g, 0, y_arr, color=color, alpha=0.18)
    ax.axvline(t0, color="gray", ls="--", lw=1.0, alpha=0.6)
    ax.set_title(title)
    ax.set_xlim(0.0, 0.40)
    ax.set_ylim(bottom=0, top=y_arr.max() * 1.18)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)

axs_pg[1, 2].set_visible(False)
axs_pg[0, 2].tick_params(labelbottom=True)
axs_pg[1, 0].set_xlabel("Time (ps)")
axs_pg[1, 1].set_xlabel("Time (ps)")
axs_pg[0, 2].set_xlabel("Time (ps)")
axs_pg[0, 0].set_ylabel(r"Drive Amplitude $\eta(t)$ ($\mathrm{ps}^{-1}$)")
axs_pg[1, 0].set_ylabel(r"Drive Amplitude $\eta(t)$ ($\mathrm{ps}^{-1}$)")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("pulse_profiles_gaussian_comparison.png", dpi=200)
plt.show()

# ------------------------------------------------------------------------------
# 2. PLOT GAUSSIAN POPULATION DYNAMICS (WITHOUT CT)
# ------------------------------------------------------------------------------
print("Plotting Gaussian population dynamics...")
def plot_populations(ax, res, title):
    P_S10 = np.real(res.expect[0])
    P_S11 = np.real(res.expect[1])
    P_T1  = np.real(res.expect[2])
    ax.plot(tlist, P_T1,  color="#9467bd", lw=2.2, label=r"$T_1$")
    ax.plot(tlist, P_S10, color="#e6a817", lw=2.2, label=r"$S_1^0$")
    ax.plot(tlist, P_S11, color="#17a2b8", lw=2.2, label=r"$S_1^1$")
    ax.set_title(title)
    ax.set_xlim(0, 4)
    ax.grid(True, alpha=0.3)

gaussian_res_list = [
    ("1. Narrow Gaussian",     res_narrow),
    ("2. Original (Baseline)", res_orig),
    ("3. Medium Gaussian",     res_medium),
    ("4. Wide Gaussian",       res_wide),
    ("5. Very Wide Gaussian",  res_very_wide),
]

fig_pop_g, axs_pop_g = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_pop_g.suptitle("Excited-State Population Dynamics of CuPc in Device D5 (Gaussian Pulses)", y=0.98)

for ax, (title, res) in zip(axs_pop_g.flat[:5], gaussian_res_list):
    plot_populations(ax, res, title)

pop_max_g = max(np.real(res.expect[k]).max() for _, res in gaussian_res_list for k in range(3))
axs_pop_g[0, 0].set_ylim(0, pop_max_g * 1.1)

axs_pop_g[1, 2].set_visible(False)
axs_pop_g[0, 2].tick_params(labelbottom=True)
axs_pop_g[1, 0].set_xlabel("Time (ps)")
axs_pop_g[1, 1].set_xlabel("Time (ps)")
axs_pop_g[0, 2].set_xlabel("Time (ps)")
axs_pop_g[0, 0].set_ylabel("Population")
axs_pop_g[1, 0].set_ylabel("Population")
axs_pop_g[0, 2].legend(loc="upper right", frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("population_dynamics_gaussian_comparison.png", dpi=200)
plt.show()

# ------------------------------------------------------------------------------
# 3. PLOT GAUSSIAN ENERGY STORED, CHARGING POWER, AND ERGOTROPY
# ------------------------------------------------------------------------------
print("Plotting Gaussian Energy Stored, Charging Power, and Ergotropy...")
gaussian_thermo_items = [
    ("1. Narrow Gaussian",     thermo_narrow,    "#2ca02c"),
    ("2. Original (Baseline)", thermo_orig,      "#d62728"),
    ("3. Medium Gaussian",     thermo_medium,    "#ff7f0e"),
    ("4. Wide Gaussian",       thermo_wide,      "#1f77b4"),
    ("5. Very Wide Gaussian",  thermo_very_wide, "#9467bd"),
]

# 3a. Energy Stored E(t)
E_lim_g = 1.1 * 1e3 * max(th[0].max() for _, th, _ in gaussian_thermo_items)   # meV
fig_eg, axs_eg = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_eg.suptitle(r"Energy Stored $E(t) = \langle H_{\mathrm{batt}} \rangle$ per Molecule (Gaussian Pulses)", y=0.98)
for ax, (title, (E_st, _, _), color) in zip(axs_eg.flat[:5], gaussian_thermo_items):
    E_meV = E_st * 1e3
    ax.plot(tlist, E_meV, color=color, lw=2.2, label=f"$E_{{\\max}} = {E_meV.max():.1f}$ meV")
    ax.fill_between(tlist, 0, E_meV, color=color, alpha=0.14)
    ax.set_title(title)
    ax.set_xlim(0, 4)
    ax.set_ylim(0, E_lim_g)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)
axs_eg[1, 2].set_visible(False)
axs_eg[0, 2].tick_params(labelbottom=True)
axs_eg[1, 0].set_xlabel("Time (ps)")
axs_eg[1, 1].set_xlabel("Time (ps)")
axs_eg[0, 2].set_xlabel("Time (ps)")
axs_eg[0, 0].set_ylabel("Energy Stored (meV)")
axs_eg[1, 0].set_ylabel("Energy Stored (meV)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("energy_stored_gaussian_comparison.png", dpi=200)
plt.show()

# 3b. Charging Power P(t)
fig_pg, axs_pwg = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)
fig_pg.suptitle(r"Instantaneous Charging Power $P(t) = dE(t)/dt$ per Molecule (Gaussian Pulses)", y=0.98)
for ax, (title, (_, P_ch, _), color) in zip(axs_pwg.flat[:5], gaussian_thermo_items):
    ax.plot(tlist, P_ch, color=color, lw=2.2, label=f"$P_{{\\max}} = {P_ch.max():.2f}$ eV/ps")
    ax.fill_between(tlist, 0, P_ch, where=(P_ch >= 0), color=color, alpha=0.18)
    ax.axhline(0, color="gray", ls="--", lw=0.9, alpha=0.7)
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)
axs_pwg[1, 2].set_visible(False)
axs_pwg[0, 2].tick_params(labelbottom=True)
axs_pwg[1, 0].set_xlabel("Time (ps)")
axs_pwg[1, 1].set_xlabel("Time (ps)")
axs_pwg[0, 2].set_xlabel("Time (ps)")
axs_pwg[0, 0].set_ylabel("Charging Power (eV / ps)")
axs_pwg[1, 0].set_ylabel("Charging Power (eV / ps)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("charging_power_gaussian_comparison.png", dpi=200)
plt.show()

# 3c. Ergotropy (Extractable Work)
fig_ergg, axs_ergg = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_ergg.suptitle(r"Ergotropy (Extractable Work) $\mathcal{E}(t)$ vs. Stored Energy $E(t)$ (Gaussian Pulses)", y=0.98)
for ax, (title, (E_st, _, Erg), color) in zip(axs_ergg.flat[:5], gaussian_thermo_items):
    E_meV = E_st * 1e3
    Erg_meV = Erg * 1e3
    ax.plot(tlist, E_meV, color="gray", ls="--", lw=1.6, alpha=0.75, label="Stored Energy $E(t)$")
    ax.plot(tlist, Erg_meV, color=color, lw=2.3, label=f"Ergotropy ($\\mathcal{{E}}_{{\\max}} = {Erg_meV.max():.1f}$ meV)")
    ax.fill_between(tlist, 0, Erg_meV, color=color, alpha=0.20)
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, E_lim_g)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.2)
axs_ergg[1, 2].set_visible(False)
axs_ergg[0, 2].tick_params(labelbottom=True)
axs_ergg[1, 0].set_xlabel("Time (ps)")
axs_ergg[1, 1].set_xlabel("Time (ps)")
axs_ergg[0, 2].set_xlabel("Time (ps)")
axs_ergg[0, 0].set_ylabel("Energy / Ergotropy (meV)")
axs_ergg[1, 0].set_ylabel("Energy / Ergotropy (meV)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("ergotropy_gaussian_comparison.png", dpi=200)
plt.show()
print("All Gaussian plots saved successfully.")

# ------------------------------------------------------------------------------
# 4. SAVE GAUSSIAN SIMULATION RESULTS
# ------------------------------------------------------------------------------
print("\nSaving Gaussian simulation results...")
eV_to_J = 1.602176634e-19
ps_to_s = 1.0e-12

gaussian_cases_data = [
    ("Narrow Gaussian",    res_narrow,    thermo_narrow),
    ("Original Baseline",  res_orig,      thermo_orig),
    ("Medium Gaussian",    res_medium,    thermo_medium),
    ("Wide Gaussian",      res_wide,      thermo_wide),
    ("Very Wide Gaussian", res_very_wide, thermo_very_wide),
]

save_dict_g = {"tlist": tlist}
csv_columns_g = [tlist]
csv_headers_g = ["time_ps"]

for name, res, (E_st, P_ch, Erg) in gaussian_cases_data:
    P_S10 = np.real(res.expect[0])
    P_S11 = np.real(res.expect[1])
    P_T1  = np.real(res.expect[2])
    n_ph  = np.real(res.expect[3])
    
    E_meV = E_st * 1e3
    Erg_meV = Erg * 1e3
    E_pass_meV = np.maximum(0.0, E_meV - Erg_meV)
    P_inst_total_W = np.gradient(N * E_st * eV_to_J, tlist * ps_to_s)
    
    save_dict_g[f"{name}_P_S10"] = P_S10
    save_dict_g[f"{name}_P_S11"] = P_S11
    save_dict_g[f"{name}_P_T1"]  = P_T1
    save_dict_g[f"{name}_n_ph"]  = n_ph
    save_dict_g[f"{name}_E_eV"]  = E_st
    save_dict_g[f"{name}_E_meV"] = E_meV
    save_dict_g[f"{name}_E_passive_meV"] = E_pass_meV
    save_dict_g[f"{name}_P_charge_eV_ps"] = P_ch
    save_dict_g[f"{name}_P_inst_total_W"] = P_inst_total_W
    save_dict_g[f"{name}_Ergotropy_eV"]  = Erg
    save_dict_g[f"{name}_Ergotropy_meV"] = Erg_meV
    
    tag = name.replace(" ", "_")
    csv_columns_g.extend([P_S10, P_S11, P_T1, n_ph, E_meV, P_ch, Erg_meV])
    csv_headers_g.extend([
        f"{tag}_S10", f"{tag}_S11", f"{tag}_T1", f"{tag}_n_ph",
        f"{tag}_E_meV", f"{tag}_P_eV_ps", f"{tag}_Erg_meV"
    ])

np.savez_compressed("protocols_sim_results_gaussian.npz", **save_dict_g)
print("-> Saved Gaussian simulation data to 'protocols_sim_results_gaussian.npz'")

csv_matrix_g = np.column_stack(csv_columns_g)
np.savetxt(
    "protocols_sim_results_gaussian.csv",
    csv_matrix_g,
    delimiter=",",
    header=",".join(csv_headers_g),
    comments=""
)
print("-> Saved Gaussian summary CSV to 'protocols_sim_results_gaussian.csv'")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

# ==============================================================================
# CELL 2: STEP FUNCTION PULSE VARIATIONS (5 VARIATIONS IN INCREASING TIME ORDER)
# ==============================================================================

# Ensure model & helper functions are available even if Cell 2 is run independently
if "build_model" not in globals():
    hbar = 6.582119569e-4          # eV * ps
    Delta1, Delta2, DeltaT, Deltac = 1.80, 1.98, 1.20, 1.82
    g_bare = 500e-9
    N      = 5.0e10
    G      = g_bare * np.sqrt(N)
    kappa, gamma_m, gamma_T = 29.0, 0.5, 0.01
    gamma_z, gamma_ISC      = 20.0, 5.0
    r          = 0.5
    t0         = 0.10
    sigma_orig = (35.0 / (2*np.sqrt(2*np.log(2)))) * 1e-3
    sigma_narrow = sigma_orig * 0.5
    I0 = r / (2.0 * np.sqrt(np.pi) * sigma_orig)
    M_pol = np.array([[Deltac, G, G], [G, Delta1, 0.0], [G, 0.0, Delta2]])
    nu = np.linalg.eigvalsh(M_pol)[0]
    d1, d2, dc = (Delta1 - nu)/hbar, (Delta2 - nu)/hbar, (Deltac - nu)/hbar
    Gw = G / hbar

    def build_model():
        Na, Nb = 5, 3
        Ia, Ib, I2 = qt.qeye(Na), qt.qeye(Nb), qt.qeye(2)
        def emb(op, slot):
            ops = [Ia, Ib, Ib, I2, I2, I2]
            ops[slot] = op
            return qt.tensor(*ops)
        a   = emb(qt.destroy(Na), 0)
        b1  = emb(qt.destroy(Nb), 1)
        b2  = emb(qt.destroy(Nb), 2)
        dk1 = emb(qt.destroy(2), 3)
        dk2 = emb(qt.destroy(2), 4)
        Tm  = emb(qt.destroy(2), 5)
        H0 = (dc * a.dag()*a + d1 * b1.dag()*b1 + d2 * b2.dag()*b2
              + Gw * (a.dag()*(b1 + b2) + (b1.dag() + b2.dag())*a))
        Hdrive = 1j * (a.dag() - a)
        c_ops = [np.sqrt(kappa)     * a,
                 np.sqrt(gamma_m)   * b1,
                 np.sqrt(gamma_m)   * b2,
                 np.sqrt(gamma_m)   * dk1,
                 np.sqrt(gamma_m)   * dk2,
                 np.sqrt(gamma_T)   * Tm,
                 np.sqrt(gamma_z)   * dk1.dag()*b1,
                 np.sqrt(gamma_z)   * dk2.dag()*b2,
                 np.sqrt(gamma_ISC) * Tm.dag()*b1,
                 np.sqrt(gamma_ISC) * Tm.dag()*b2,
                 np.sqrt(gamma_ISC) * Tm.dag()*dk1,
                 np.sqrt(gamma_ISC) * Tm.dag()*dk2]
        e_ops = [b1.dag()*b1 + dk1.dag()*dk1,
                 b2.dag()*b2 + dk2.dag()*dk2,
                 Tm.dag()*Tm,
                 a.dag()*a]
        H_batt = (Delta1 * (b1.dag()*b1 + dk1.dag()*dk1)
                  + Delta2 * (b2.dag()*b2 + dk2.dag()*dk2)
                  + DeltaT * Tm.dag()*Tm)
        b1_mol  = qt.tensor(qt.destroy(Nb), Ib, I2, I2, I2)
        b2_mol  = qt.tensor(Ib, qt.destroy(Nb), I2, I2, I2)
        dk1_mol = qt.tensor(Ib, Ib, qt.destroy(2), I2, I2)
        dk2_mol = qt.tensor(Ib, Ib, I2, qt.destroy(2), I2)
        Tm_mol  = qt.tensor(Ib, Ib, I2, I2, qt.destroy(2))
        H_batt_mol = (Delta1 * (b1_mol.dag()*b1_mol + dk1_mol.dag()*dk1_mol)
                      + Delta2 * (b2_mol.dag()*b2_mol + dk2_mol.dag()*dk2_mol)
                      + DeltaT * Tm_mol.dag()*Tm_mol)
        psi0 = qt.tensor(qt.basis(Na, 0), qt.basis(Nb, 0), qt.basis(Nb, 0),
                         qt.basis(2, 0), qt.basis(2, 0), qt.basis(2, 0))
        return H0, Hdrive, c_ops, e_ops, psi0, H_batt, H_batt_mol

    def compute_thermodynamics(res, tlist, eps_sorted):
        P_S10 = np.real(res.expect[0])
        P_S11 = np.real(res.expect[1])
        P_T1  = np.real(res.expect[2])
        E_stored = Delta1 * P_S10 + Delta2 * P_S11 + DeltaT * P_T1
        P_charge = np.gradient(E_stored, tlist)
        ergotropy = np.zeros(len(tlist))
        for i, state in enumerate(res.states):
            rho_mol = state.ptrace([1, 2, 3, 4, 5])
            r_vals = np.sort(np.real(rho_mol.eigenenergies()))[::-1]
            ergotropy[i] = max(0.0, E_stored[i] - np.sum(r_vals * eps_sorted))
        return E_stored, P_charge, ergotropy

    def plot_populations(ax, res, title):
        P_S10 = np.real(res.expect[0])
        P_S11 = np.real(res.expect[1])
        P_T1  = np.real(res.expect[2])
        ax.plot(tlist, P_T1,  color="#9467bd", lw=2.2, label=r"$T_1$")
        ax.plot(tlist, P_S10, color="#e6a817", lw=2.2, label=r"$S_1^0$")
        ax.plot(tlist, P_S11, color="#17a2b8", lw=2.2, label=r"$S_1^1$")
        ax.set_title(title)
        ax.set_xlim(0, 4)
        ax.grid(True, alpha=0.3)

# 5 Step function pulse duration variations (ordered by increasing total duration)
# Keeping the original 0.10 ps (100 fs) step duration + 4 wider durations
step_dur_1 = 0.10   # 1. Step 100 fs (t in [0.10, 0.20] ps) [Original Step]
step_dur_2 = 0.15   # 2. Step 150 fs (t in [0.10, 0.25] ps)
step_dur_3 = 0.20   # 3. Step 200 fs (t in [0.10, 0.30] ps)
step_dur_4 = 0.25   # 4. Step 250 fs (t in [0.10, 0.35] ps)
step_dur_5 = 0.30   # 5. Step 300 fs (t in [0.10, 0.40] ps)

# Step Drive functions (all normalized to the same total pulse energy \int |\eta(t)|^2 dt = I0)
def eta_step_1(t, **kwargs):
    return np.sqrt(I0 / step_dur_1) if t0 <= t <= (t0 + step_dur_1) else 0.0

def eta_step_2(t, **kwargs):
    return np.sqrt(I0 / step_dur_2) if t0 <= t <= (t0 + step_dur_2) else 0.0

def eta_step_3(t, **kwargs):
    return np.sqrt(I0 / step_dur_3) if t0 <= t <= (t0 + step_dur_3) else 0.0

def eta_step_4(t, **kwargs):
    return np.sqrt(I0 / step_dur_4) if t0 <= t <= (t0 + step_dur_4) else 0.0

def eta_step_5(t, **kwargs):
    return np.sqrt(I0 / step_dur_5) if t0 <= t <= (t0 + step_dur_5) else 0.0


H0, Hdrive, c_ops, e_ops, psi0, H_batt, H_batt_mol = build_model()
eps_batt_sorted = np.sort(np.real(H_batt_mol.eigenenergies()))

tlist = np.linspace(0.0, 4.0, 801)     # ps
opts = {"atol": 1e-9, "rtol": 1e-7, "nsteps": 1000000, "store_states": True,
        "max_step": sigma_narrow / 5}   # ~1.5 fs

print("Running 1/5: Step Function (Duration = 100 fs, Original)...")
res_step_1 = qt.mesolve([H0, [Hdrive, eta_step_1]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 2/5: Step Function (Duration = 150 fs)...")
res_step_2 = qt.mesolve([H0, [Hdrive, eta_step_2]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 3/5: Step Function (Duration = 200 fs)...")
res_step_3 = qt.mesolve([H0, [Hdrive, eta_step_3]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 4/5: Step Function (Duration = 250 fs)...")
res_step_4 = qt.mesolve([H0, [Hdrive, eta_step_4]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Running 5/5: Step Function (Duration = 300 fs)...")
res_step_5 = qt.mesolve([H0, [Hdrive, eta_step_5]], psi0, tlist, c_ops, e_ops=e_ops, options=opts)

print("Computing thermodynamic observables for Step Function variations...")
thermo_step_1 = compute_thermodynamics(res_step_1, tlist, eps_batt_sorted)
thermo_step_2 = compute_thermodynamics(res_step_2, tlist, eps_batt_sorted)
thermo_step_3 = compute_thermodynamics(res_step_3, tlist, eps_batt_sorted)
thermo_step_4 = compute_thermodynamics(res_step_4, tlist, eps_batt_sorted)
thermo_step_5 = compute_thermodynamics(res_step_5, tlist, eps_batt_sorted)

# ------------------------------------------------------------------------------
# 1. PLOT STEP FUNCTION LASER PULSE PROFILES
# ------------------------------------------------------------------------------
print("Plotting Step Function laser pulse profiles...")
t_pulse_s = np.linspace(0.0, 0.45, 1000)

step_pulse_list = [
    ("1. Step Function (100 fs)", [eta_step_1(t) for t in t_pulse_s], "#ff7f0e", f"Duration = {step_dur_1*1e3:.0f}.0 fs ($t \\in [{t0:.2f}, {t0+step_dur_1:.2f}]$ ps)"),
    ("2. Step Function (150 fs)", [eta_step_2(t) for t in t_pulse_s], "#2ca02c", f"Duration = {step_dur_2*1e3:.0f}.0 fs ($t \\in [{t0:.2f}, {t0+step_dur_2:.2f}]$ ps)"),
    ("3. Step Function (200 fs)", [eta_step_3(t) for t in t_pulse_s], "#1f77b4", f"Duration = {step_dur_3*1e3:.0f}.0 fs ($t \\in [{t0:.2f}, {t0+step_dur_3:.2f}]$ ps)"),
    ("4. Step Function (250 fs)", [eta_step_4(t) for t in t_pulse_s], "#d62728", f"Duration = {step_dur_4*1e3:.0f}.0 fs ($t \\in [{t0:.2f}, {t0+step_dur_4:.2f}]$ ps)"),
    ("5. Step Function (300 fs)", [eta_step_5(t) for t in t_pulse_s], "#9467bd", f"Duration = {step_dur_5*1e3:.0f}.0 fs ($t \\in [{t0:.2f}, {t0+step_dur_5:.2f}]$ ps)"),
]

fig_pulse_s, axs_ps = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)
fig_pulse_s.suptitle(r"Step Function Laser Pulse Profiles $\eta(t)$ (Energy-Normalized: $\int |\eta(t)|^2\,dt = I_0$)", y=0.98)

for ax, (title, y_vals, color, label_str) in zip(axs_ps.flat[:5], step_pulse_list):
    y_arr = np.array(y_vals)
    ax.plot(t_pulse_s, y_arr, color=color, lw=2.3, label=label_str)
    ax.fill_between(t_pulse_s, 0, y_arr, color=color, alpha=0.18)
    ax.axvline(t0, color="gray", ls="--", lw=1.0, alpha=0.6)
    ax.set_title(title)
    ax.set_xlim(0.0, 0.45)
    ax.set_ylim(bottom=0, top=y_arr.max() * 1.18)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)

axs_ps[1, 2].set_visible(False)
axs_ps[0, 2].tick_params(labelbottom=True)
axs_ps[1, 0].set_xlabel("Time (ps)")
axs_ps[1, 1].set_xlabel("Time (ps)")
axs_ps[0, 2].set_xlabel("Time (ps)")
axs_ps[0, 0].set_ylabel(r"Drive Amplitude $\eta(t)$ ($\mathrm{ps}^{-1}$)")
axs_ps[1, 0].set_ylabel(r"Drive Amplitude $\eta(t)$ ($\mathrm{ps}^{-1}$)")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("pulse_profiles_step_comparison.png", dpi=200)
plt.show()

# ------------------------------------------------------------------------------
# 2. PLOT STEP FUNCTION POPULATION DYNAMICS (WITHOUT CT)
# ------------------------------------------------------------------------------
print("Plotting Step Function population dynamics...")
step_res_list = [
    ("1. Step Function (100 fs)", res_step_1),
    ("2. Step Function (150 fs)", res_step_2),
    ("3. Step Function (200 fs)", res_step_3),
    ("4. Step Function (250 fs)", res_step_4),
    ("5. Step Function (300 fs)", res_step_5),
]

fig_pop_s, axs_pop_s = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_pop_s.suptitle("Excited-State Population Dynamics of CuPc in Device D5 (Step Function Pulses)", y=0.98)

for ax, (title, res) in zip(axs_pop_s.flat[:5], step_res_list):
    plot_populations(ax, res, title)

pop_max_s = max(np.real(res.expect[k]).max() for _, res in step_res_list for k in range(3))
axs_pop_s[0, 0].set_ylim(0, pop_max_s * 1.1)

axs_pop_s[1, 2].set_visible(False)
axs_pop_s[0, 2].tick_params(labelbottom=True)
axs_pop_s[1, 0].set_xlabel("Time (ps)")
axs_pop_s[1, 1].set_xlabel("Time (ps)")
axs_pop_s[0, 2].set_xlabel("Time (ps)")
axs_pop_s[0, 0].set_ylabel("Population")
axs_pop_s[1, 0].set_ylabel("Population")
axs_pop_s[0, 2].legend(loc="upper right", frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("population_dynamics_step_comparison.png", dpi=200)
plt.show()

# ------------------------------------------------------------------------------
# 3. PLOT STEP FUNCTION ENERGY STORED, CHARGING POWER, AND ERGOTROPY
# ------------------------------------------------------------------------------
print("Plotting Step Function Energy Stored, Charging Power, and Ergotropy...")
step_thermo_items = [
    ("1. Step Function (100 fs)", thermo_step_1, "#ff7f0e"),
    ("2. Step Function (150 fs)", thermo_step_2, "#2ca02c"),
    ("3. Step Function (200 fs)", thermo_step_3, "#1f77b4"),
    ("4. Step Function (250 fs)", thermo_step_4, "#d62728"),
    ("5. Step Function (300 fs)", thermo_step_5, "#9467bd"),
]

# 3a. Energy Stored E(t)
E_lim_s = 1.1 * 1e3 * max(th[0].max() for _, th, _ in step_thermo_items)   # meV
fig_es, axs_es = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_es.suptitle(r"Energy Stored $E(t) = \langle H_{\mathrm{batt}} \rangle$ per Molecule (Step Function Pulses)", y=0.98)
for ax, (title, (E_st, _, _), color) in zip(axs_es.flat[:5], step_thermo_items):
    E_meV = E_st * 1e3
    ax.plot(tlist, E_meV, color=color, lw=2.2, label=f"$E_{{\\max}} = {E_meV.max():.1f}$ meV")
    ax.fill_between(tlist, 0, E_meV, color=color, alpha=0.14)
    ax.set_title(title)
    ax.set_xlim(0, 4)
    ax.set_ylim(0, E_lim_s)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)
axs_es[1, 2].set_visible(False)
axs_es[0, 2].tick_params(labelbottom=True)
axs_es[1, 0].set_xlabel("Time (ps)")
axs_es[1, 1].set_xlabel("Time (ps)")
axs_es[0, 2].set_xlabel("Time (ps)")
axs_es[0, 0].set_ylabel("Energy Stored (meV)")
axs_es[1, 0].set_ylabel("Energy Stored (meV)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("energy_stored_step_comparison.png", dpi=200)
plt.show()

# 3b. Charging Power P(t)
fig_ps, axs_pws = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)
fig_ps.suptitle(r"Instantaneous Charging Power $P(t) = dE(t)/dt$ per Molecule (Step Function Pulses)", y=0.98)
for ax, (title, (_, P_ch, _), color) in zip(axs_pws.flat[:5], step_thermo_items):
    ax.plot(tlist, P_ch, color=color, lw=2.2, label=f"$P_{{\\max}} = {P_ch.max():.2f}$ eV/ps")
    ax.fill_between(tlist, 0, P_ch, where=(P_ch >= 0), color=color, alpha=0.18)
    ax.axhline(0, color="gray", ls="--", lw=0.9, alpha=0.7)
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)
axs_pws[1, 2].set_visible(False)
axs_pws[0, 2].tick_params(labelbottom=True)
axs_pws[1, 0].set_xlabel("Time (ps)")
axs_pws[1, 1].set_xlabel("Time (ps)")
axs_pws[0, 2].set_xlabel("Time (ps)")
axs_pws[0, 0].set_ylabel("Charging Power (eV / ps)")
axs_pws[1, 0].set_ylabel("Charging Power (eV / ps)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("charging_power_step_comparison.png", dpi=200)
plt.show()

# 3c. Ergotropy (Extractable Work)
fig_ergs, axs_ergs = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True, sharey=True)
fig_ergs.suptitle(r"Ergotropy (Extractable Work) $\mathcal{E}(t)$ vs. Stored Energy $E(t)$ (Step Function Pulses)", y=0.98)
for ax, (title, (E_st, _, Erg), color) in zip(axs_ergs.flat[:5], step_thermo_items):
    E_meV = E_st * 1e3
    Erg_meV = Erg * 1e3
    ax.plot(tlist, E_meV, color="gray", ls="--", lw=1.6, alpha=0.75, label="Stored Energy $E(t)$")
    ax.plot(tlist, Erg_meV, color=color, lw=2.3, label=f"Ergotropy ($\\mathcal{{E}}_{{\\max}} = {Erg_meV.max():.1f}$ meV)")
    ax.fill_between(tlist, 0, Erg_meV, color=color, alpha=0.20)
    ax.set_title(title)
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, E_lim_s)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=True, fontsize=8.2)
axs_ergs[1, 2].set_visible(False)
axs_ergs[0, 2].tick_params(labelbottom=True)
axs_ergs[1, 0].set_xlabel("Time (ps)")
axs_ergs[1, 1].set_xlabel("Time (ps)")
axs_ergs[0, 2].set_xlabel("Time (ps)")
axs_ergs[0, 0].set_ylabel("Energy / Ergotropy (meV)")
axs_ergs[1, 0].set_ylabel("Energy / Ergotropy (meV)")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("ergotropy_step_comparison.png", dpi=200)
plt.show()
print("All Step Function plots saved successfully.")

# ------------------------------------------------------------------------------
# 4. SAVE STEP FUNCTION SIMULATION RESULTS
# ------------------------------------------------------------------------------
print("\nSaving Step Function simulation results...")
eV_to_J = 1.602176634e-19
ps_to_s = 1.0e-12

step_cases_data = [
    ("Step 100fs", res_step_1, thermo_step_1),
    ("Step 150fs", res_step_2, thermo_step_2),
    ("Step 200fs", res_step_3, thermo_step_3),
    ("Step 250fs", res_step_4, thermo_step_4),
    ("Step 300fs", res_step_5, thermo_step_5),
]

save_dict_s = {"tlist": tlist}
csv_columns_s = [tlist]
csv_headers_s = ["time_ps"]

for name, res, (E_st, P_ch, Erg) in step_cases_data:
    P_S10 = np.real(res.expect[0])
    P_S11 = np.real(res.expect[1])
    P_T1  = np.real(res.expect[2])
    n_ph  = np.real(res.expect[3])
    
    E_meV = E_st * 1e3
    Erg_meV = Erg * 1e3
    E_pass_meV = np.maximum(0.0, E_meV - Erg_meV)
    P_inst_total_W = np.gradient(N * E_st * eV_to_J, tlist * ps_to_s)
    
    save_dict_s[f"{name}_P_S10"] = P_S10
    save_dict_s[f"{name}_P_S11"] = P_S11
    save_dict_s[f"{name}_P_T1"]  = P_T1
    save_dict_s[f"{name}_n_ph"]  = n_ph
    save_dict_s[f"{name}_E_eV"]  = E_st
    save_dict_s[f"{name}_E_meV"] = E_meV
    save_dict_s[f"{name}_E_passive_meV"] = E_pass_meV
    save_dict_s[f"{name}_P_charge_eV_ps"] = P_ch
    save_dict_s[f"{name}_P_inst_total_W"] = P_inst_total_W
    save_dict_s[f"{name}_Ergotropy_eV"]  = Erg
    save_dict_s[f"{name}_Ergotropy_meV"] = Erg_meV
    
    tag = name.replace(" ", "_")
    csv_columns_s.extend([P_S10, P_S11, P_T1, n_ph, E_meV, P_ch, Erg_meV])
    csv_headers_s.extend([
        f"{tag}_S10", f"{tag}_S11", f"{tag}_T1", f"{tag}_n_ph",
        f"{tag}_E_meV", f"{tag}_P_eV_ps", f"{tag}_Erg_meV"
    ])

np.savez_compressed("protocols_sim_results_step.npz", **save_dict_s)
print("-> Saved Step Function simulation data to 'protocols_sim_results_step.npz'")

csv_matrix_s = np.column_stack(csv_columns_s)
np.savetxt(
    "protocols_sim_results_step.csv",
    csv_matrix_s,
    delimiter=",",
    header=",".join(csv_headers_s),
    comments=""
)
print("-> Saved Step Function summary CSV to 'protocols_sim_results_step.csv'")